# Archive version, NOT EXECUTED

In [3]:
suppressPackageStartupMessages({
  library(ENmix)
  library(IlluminaHumanMethylationEPICmanifest)
  library(IlluminaHumanMethylationEPICanno.ilm10b2.hg19)
  library(parallel)
  library(methylCIPHER)
    library(glmnet);
    library(EpiDISH);
    library(dnaMethyAge)
})

In [72]:
# Reading phenotypic data
mf = "/gpfs/gibbs/pi/montalvo-ortiz/epigenomics/braindata/collaborations/consuelobassdata/brain_dataset/meta_BA9_brainbank_methylation_sub_3-29-22.csv"
mf = read.csv(mf)
mf$name <- paste0("BRA_", mf$Sentrix_ID)
head(mf$name)

[1] "BRA_201465920012_R01C01" "BRA_201465920012_R02C01"
[3] "BRA_201465920012_R03C01" "BRA_201465920012_R04C01"
[5] "BRA_201465920012_R05C01" "BRA_201465920012_R06C01"

In [12]:
# setting cores
num_threads <- detectCores() - 1
print(num_threads)
# set working directory
wkdir="/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome"
setwd(wkdir)

[1] 47


In [14]:
path="/gpfs/gibbs/pi/montalvo-ortiz/epigenomics/braindata/collaborations/consuelobassdata/brain_dataset/idat_brain"
rgSet <- readidat(path = path,
                 recursive = TRUE,
                 force = TRUE)

[readidat] Found 123 files with suffix _Grn.idat

[readidat] Found 123 files with suffix _Red.idat



BRA_201465920012_R01C01 BRA_201465920012_R02C01 BRA_201465920012_R03C01 
                1051943                 1051943                 1051943 
BRA_201465920012_R04C01 BRA_201465920012_R05C01 BRA_201465920012_R06C01 
                1051943                 1051943                 1051943 
BRA_201465920012_R07C01 BRA_201465920012_R08C01 BRA_201465920048_R01C01 
                1051943                 1051943                 1051943 
BRA_201465920048_R02C01 BRA_201465920048_R03C01 BRA_201465920048_R04C01 
                1051943                 1051943                 1051943 
BRA_201465920048_R05C01 BRA_201465920048_R06C01 BRA_201465920048_R07C01 
                1051943                 1051943                 1051943 
BRA_201465920048_R08C01 BRA_201532190145_R01C01 BRA_201532190145_R02C01 
                1051943                 1051815                 1051815 
BRA_201532190145_R03C01 BRA_201532190145_R04C01 BRA_201532190145_R05C01 
                1051815                 1051815    

Arrays with different sizes are forced to merge together


[readidat] Red Green Channel contains 1008711 probes and 123 Samples



In [35]:
# Perform QC
qc<-QCinfo(rgSet)
#QCinfo returns "detP","nbead","bisul","badsample","badCpG","outlier_sample"
#Samples with low quality data
qc$badsample
#CpGs with low quality data
qc$badCpG
#outlier samples
qc$outlier_sample

7  samples with percentage of low quality CpG value greater than  0.05  or bisulfite intensity less than  9777.351 
44801  CpGs with percentage of low quality value greater than  0.05 
Ploting qc_sample.jpg ...Done
Ploting qc_CpG.jpg ...Done
Identifying ourlier samples based on beta or total intensity values...
After excluding low quality samples and CpGs
0  samples are outliers based on averaged total intensity value 
2  samples are outliers in beta value distribution 
2  outlier samples were added into badsample list
Ploting freqpolygon_beta_beforeQC.jpg ...Done
Ploting freqpolygon_beta_afterQC.jpg ...Done


[1] "BRA_201465920012_R07C01" "BRA_201532190145_R07C01"
[3] "BRA_202787550059_R02C01" "BRA_202787550059_R08C01"
[5] "BRA_202787550060_R03C01" "BRA_202787550060_R06C01"
[7] "BRA_205096370104_R06C01" "BRA_205016910070_R08C01"
[9] "BRA_205096370104_R08C01"

[1] "cg00007036"       "cg00008800"       "cg00013655"      
    [4] "cg00028598"       "cg00033643"       "cg00039794"      
    [7] "cg00055165"       "cg00058299"       "cg00084888"      
   [10] "cg00090103"       "cg00090813"       "cg00099023"      
   [13] "cg00160667"       "cg00161970"       "cg00162673"      
   [16] "cg00164462"       "cg00179892"       "cg00199846"      
   [19] "cg00207100"       "cg00212031"       "cg00213748"      
   [22] "cg00214611"       "cg00223863"       "cg00225905"      
   [25] "cg00275181"       "cg00285060"       "cg00313642"      
   [28] "cg00332802"       "cg00334798"       "cg00343349"      
   [31] "cg00356183"       "cg00399938"       "cg00401101"      
   [34] "cg00416747"       "cg00418844"       "cg00428160"      
   [37] "cg00433224"       "cg00448891"       "cg00455876"      
   [40] "cg00465927"       "cg00472375"       "cg00501779"      
   [43] "cg00503383"       "cg00531823"       "cg00536058"      
   [46] "cg00611675"       "cg00632914"       "cg00637791"      
   [49] "cg00659991"       "cg00678354"       "cg00680875"      
   [52] "cg00729564"       "cg00742920"       "cg00748431"      
   [55] "cg00771642"       "cg00781110"       "cg00795341"      
   [58] "cg00798886"       "cg00803088"       "cg00808935"      
   [61] "cg00817367"       "cg00843105"       "cg00857557"      
   [64] "cg00859129"       "cg00864568"       "cg00870694"      
   [67] "cg00877553"       "cg00878516"       "cg00898282"      
   [70] "cg00917899"       "cg00922825"       "cg00974689"      
   [73] "cg00988678"       "cg01002341"       "cg01006609"      
   [76] "cg01019333"       "cg01036409"       "cg01040461"      
   [79] "cg01061843"       "cg01068452"       "cg01070148"      
   [82] "cg01094544"       "cg01096398"       "cg01097907"      
   [85] "cg01111656"       "cg01145653"       "cg01153979"      
   [88] "cg01166071"       "cg01166145"       "cg01168833"      
   [91] "cg01177470"       "cg01188170"       "cg01191902"      
   [94] "cg01217666"       "cg01224520"       "cg01230890"      
   [97] "cg01238044"       "cg01239717"       "cg01241624"      
  [100] "cg01266264"       "cg01270299"       "cg01295782"      
  [103] "cg01306183"       "cg01351037"       "cg01370437"      
  [106] "cg01392253"       "cg01395800"       "cg01399914"      
  [109] "cg01403309"       "cg01413492"       "cg01414268"      
  [112] "cg01445689"       "cg01446203"       "cg01471886"      
  [115] "cg01483252"       "cg01483824"       "cg01485157"      
  [118] "cg01504215"       "cg01541629"       "cg01547069"      
  [121] "cg01554235"       "cg01556227"       "cg01588748"      
  [124] "cg01590424"       "cg01593834"       "cg01594514"      
  [127] "cg01614020"       "cg01615850"       "cg01653080"      
  [130] "cg01659202"       "cg01659459"       "cg01663745"      
  [133] "cg01666550"       "cg01678753"       "cg01680010"      
  [136] "cg01683654"       "cg01703419"       "cg01731204"      
  [139] "cg01742905"       "cg01748414"       "cg01755467"      
  [142] "cg01778153"       "cg01788113"       "cg01789160"      
  [145] "cg01793374"       "cg01807421"       "cg01816482"      
  [148] "cg01826863"       "cg01835144"       "cg01842890"      
  [151] "cg01845244"       "cg01850767"       "cg01856069"      
  [154] "cg01866022"       "cg01874886"       "cg01878345"      
  [157] "cg01891497"       "cg01923218"       "cg01934296"      
  [160] "cg01938978"       "cg01941018"       "cg01957799"      
  [163] "cg01967073"       "cg01994308"       "cg01995145"      
  [166] "cg02000145"       "cg02005549"       "cg02011394"      
  [169] "cg02012576"       "cg02020650"       "cg02049949"      
  [172] "cg02050593"       "cg02069150"       "cg02084758"      
  [175] "cg02090806"       "cg02125316"       "cg02129337"      
  [178] "cg02153340"       "cg02155627"       "cg02157755"      
  [181] "cg02183105"       "cg02190610"       "cg02202022"      
  [184] "cg02227969"       "cg02230495"

[1] "BRA_205016910070_R08C01" "BRA_205096370104_R08C01"

In [37]:
# First estimate Betas
meta = getmeth(rgSet)
beta = getB(meta)

# filter low quality and outlier values, remove rows and
# columns with too many (rthre=0.05,cthre=0.05, 5% in default) missing values,
# and then do imputation
#b3=qcfilter(beta, qcscore=qc, rmcr=TRUE, rthre=0.05,
#              cthre=0.05, impute=TRUE)

In [38]:
# Perform SVA analysis
sva <- ctrlsva(rgSet)
sva = as.data.frame(sva)
sva$name = rownames(sva)

8  surrogate variables explain  95.37677 % of 
    data variation


In [18]:
celltype = estimateCellProp(userdata=rgSet,
         refdata="FlowSorted.DLPFC.450k",
         nonnegative = TRUE, normalize=TRUE)

Loading required package: IlluminaHumanMethylation450kmanifest



In [73]:
# Merge celltypes
celltype$name = celltype$Sample_Name
pheno = merge(mf, celltype, by = c('name'))
pheno = merge(pheno, sva, by = c('name'))

In [44]:
# Estimate clocks using methylCIPHER
# Need to transpose the matrix, samples in columns and cpg in rows
beta = as.data.frame(t(beta))
calcCoreClocks(beta, pheno)

Please remember to cite the core Clocks you have used! Please refer to the README.md file for assistance.



name,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,pH,Cause_of_Death,⋯,PC3,PC4,PC5,PC6,PC7,PC8,Hannum,Horvath1,PhenoAge,epiTOC2
<chr>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BRA_201465920012_R01C01,67917,201465920012_R01C01,2018,Hispanic,Male,44,32.67,5.91,acute carisoprodol toxicity,⋯,-6665.8843,1293.0304,-5005.23120,2870.01084,-3640.1573,355.5046,18.73410,43.97707,-3.2600031,738.4980
BRA_201465920012_R02C01,67927,201465920012_R02C01,2018,Black,Female,32,31.57,6.56,undetermined,⋯,-4262.4880,-3302.8470,-1485.83904,-6463.84352,-1715.2361,2302.3781,11.48559,43.30013,2.1732831,855.6126
BRA_201465920012_R03C01,67928,201465920012_R03C01,2018,Black,Male,53,28.65,6.79,acute toxicity of cocaine and ethanol,⋯,-988.6257,-8945.0667,-5556.49206,1126.86676,-4362.8206,1257.2628,21.34757,49.79894,-2.0383794,1039.6543
BRA_201465920012_R04C01,67929,201465920012_R04C01,2018,White,Female,36,28.20,6.80,toxic effects of methadone,⋯,-2545.9918,-13250.4964,-2082.82146,-4780.32599,-5395.5371,-1540.2280,20.43901,37.99854,1.9195821,951.4873
BRA_201465920012_R05C01,67930,201465920012_R05C01,2018,White,Male,48,31.30,6.44,GSW,⋯,-2974.7844,-10869.5795,-5945.75523,-1059.95173,-6963.1584,2966.0035,27.52843,44.14654,-10.6833841,1037.1281
BRA_201465920012_R06C01,67931,201465920012_R06C01,2018,White,Male,52,31.92,6.38,"cardiovascular disease, not opioid",⋯,-2278.0445,-14641.3552,-4522.12854,-3.86258,-3440.9959,1329.0686,29.39186,49.12554,-1.9751743,994.0742
BRA_201465920012_R07C01,67941,201465920012_R07C01,2018,White,Male,64,26.35,6.40,ruptured AAA,⋯,-2967.0714,-10692.0569,1858.70118,1448.24412,-2024.6519,-2061.6679,27.32664,50.02112,9.6122992,1138.7123
BRA_201465920012_R08C01,67943,201465920012_R08C01,2018,White,Male,33,10.57,6.72,"multi overdose: benzodiazepine, amphetamine, cocaine, EtOH, and opioid",⋯,-5088.2332,11016.2750,-4462.38435,1794.73226,-820.1338,-1034.7168,8.95954,39.38449,0.8140172,601.3421
BRA_201465920048_R01C01,67944,201465920048_R01C01,2018,White,Female,50,9.55,6.63,sepsis,⋯,-3284.8649,187.7708,-1597.55280,-3680.88142,-1699.2859,590.2310,22.09779,47.38520,0.6649717,1065.5823


In [59]:
# Preparing data for stochastic clock
beta_m = as.matrix(t(beta))
age = as.vector(pheno$Age)
dim(beta_m)
dim(age)
head(colnames(beta_m))  # Should be sample IDs

[1] 865918    123

NULL

[1] "BRA_201465920012_R01C01" "BRA_201465920012_R02C01"
[3] "BRA_201465920012_R03C01" "BRA_201465920012_R04C01"
[5] "BRA_201465920012_R05C01" "BRA_201465920012_R06C01"

In [60]:
# Running stocastic clocks
clockpath = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/03scripts/01epigenome/glmStocALL.Rd"

# This is the function for stochastic clocks
RunStochClocks <- function(data.m,ages.v=NULL,refM.m=NULL){
 load(clockpath); ## load in stochastic clock information
 estF.m <- NULL;
 if(!is.null(refM.m)){
   estF.m <- epidish(data.m,ref=refM.m,method="RPC",maxit=500)$est;
 }
 predMage.lv <- list();
 eaa.lv <- list();
 iaa.lv <- list();
 for(c in 1:length(glmStocALL.lo)){
  glm.o <- glmStocALL.lo[[c]];
  coef.m <- coef(glm.o);
  intC <- coef.m[1,ncol(coef.m)]; ### intercept term
  coef.v <- coef.m[-1,ncol(coef.m)]; ### estimated regression coefficients
  commonCpGs.v <- intersect(names(coef.v),rownames(data.m));
  rep.idx <- match(commonCpGs.v,names(coef.v));
  map.idx <- match(commonCpGs.v,rownames(data.m));
  predMage.lv[[c]] <- as.vector(intC + matrix(coef.v[rep.idx],nrow=1) %*% data.m[map.idx,]);
  if(!is.null(ages.v)){
     eaa.lv[[c]] <- lm(predMage.lv[[c]] ~ ages.v)$res;     
     if(!is.null(refM.m)){
      iaa.lv[[c]] <- lm(predMage.lv[[c]] ~ ages.v + estF.m)$res;
     }
  }
 }
 return(list(mage=predMage.lv,eaa=eaa.lv,iaa=iaa.lv,estF=estF.m));
}

# Running function
sclocks = RunStochClocks(beta_m, age)

In [75]:
# Add sample IDs first (these are the column names of beta)
sample_ids <- colnames(beta_m)

mage_df <- as.data.frame(do.call(cbind, sclocks$mage))
colnames(mage_df) <- paste0("StochClock", seq_along(sclocks$mage))
mage_df$name <- sample_ids

eaa_df <- as.data.frame(do.call(cbind, sclocks$eaa))
colnames(eaa_df) <- paste0("EAA_StochClock", seq_along(sclocks$eaa))
eaa_df$name <- sample_ids

# Merge side by side by 'name'
combined_df <- merge(mage_df, eaa_df, by = "name")
combined_df
pheno = merge(pheno, mage_df, by = c('name'))
pheno

name,StochClock1,StochClock2,StochClock3,EAA_StochClock1,EAA_StochClock2,EAA_StochClock3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BRA_201465920012_R01C01,39.59430,22.391614,57.06745,3.9628005,2.6042783,2.7030551
BRA_201465920012_R02C01,34.52284,24.065296,60.98985,1.9302871,10.0491274,6.9936651
BRA_201465920012_R03C01,43.22477,16.941543,52.77841,5.3140622,-7.1741671,-1.8621434
BRA_201465920012_R04C01,31.90022,17.266903,56.09438,-1.7053117,1.3270119,1.9754579
BRA_201465920012_R05C01,39.15640,19.215417,53.88648,2.5119189,-2.4956410,-0.6006489
BRA_201465920012_R06C01,32.98630,22.640571,57.32405,-4.6711597,-0.9942085,2.7141787
BRA_201465920012_R07C01,39.03303,26.391507,60.44997,-1.6633780,-3.0144396,5.4718892
BRA_201465920012_R08C01,33.02133,22.930111,53.04305,0.1755288,8.4330115,-0.9838217
BRA_201465920048_R01C01,41.99910,24.106571,45.90228,4.8481276,1.4336525,-8.6462190


name,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,pH,Cause_of_Death,⋯,PC5,PC6,PC7,PC8,StochClock1.x,StochClock2.x,StochClock3.x,StochClock1.y,StochClock2.y,StochClock3.y
<chr>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
BRA_201465920012_R01C01,67917,201465920012_R01C01,2018,Hispanic,Male,44,32.67,5.91,acute carisoprodol toxicity,⋯,-5005.23120,2870.01084,-3640.1573,355.5046,39.59430,22.391614,57.06745,39.59430,22.391614,57.06745
BRA_201465920012_R02C01,67927,201465920012_R02C01,2018,Black,Female,32,31.57,6.56,undetermined,⋯,-1485.83904,-6463.84352,-1715.2361,2302.3781,34.52284,24.065296,60.98985,34.52284,24.065296,60.98985
BRA_201465920012_R03C01,67928,201465920012_R03C01,2018,Black,Male,53,28.65,6.79,acute toxicity of cocaine and ethanol,⋯,-5556.49206,1126.86676,-4362.8206,1257.2628,43.22477,16.941543,52.77841,43.22477,16.941543,52.77841
BRA_201465920012_R04C01,67929,201465920012_R04C01,2018,White,Female,36,28.20,6.80,toxic effects of methadone,⋯,-2082.82146,-4780.32599,-5395.5371,-1540.2280,31.90022,17.266903,56.09438,31.90022,17.266903,56.09438
BRA_201465920012_R05C01,67930,201465920012_R05C01,2018,White,Male,48,31.30,6.44,GSW,⋯,-5945.75523,-1059.95173,-6963.1584,2966.0035,39.15640,19.215417,53.88648,39.15640,19.215417,53.88648
BRA_201465920012_R06C01,67931,201465920012_R06C01,2018,White,Male,52,31.92,6.38,"cardiovascular disease, not opioid",⋯,-4522.12854,-3.86258,-3440.9959,1329.0686,32.98630,22.640571,57.32405,32.98630,22.640571,57.32405
BRA_201465920012_R07C01,67941,201465920012_R07C01,2018,White,Male,64,26.35,6.40,ruptured AAA,⋯,1858.70118,1448.24412,-2024.6519,-2061.6679,39.03303,26.391507,60.44997,39.03303,26.391507,60.44997
BRA_201465920012_R08C01,67943,201465920012_R08C01,2018,White,Male,33,10.57,6.72,"multi overdose: benzodiazepine, amphetamine, cocaine, EtOH, and opioid",⋯,-4462.38435,1794.73226,-820.1338,-1034.7168,33.02133,22.930111,53.04305,33.02133,22.930111,53.04305
BRA_201465920048_R01C01,67944,201465920048_R01C01,2018,White,Female,50,9.55,6.63,sepsis,⋯,-1597.55280,-3680.88142,-1699.2859,590.2310,41.99910,24.106571,45.90228,41.99910,24.106571,45.90228


In [77]:
# Save specific objects to an RData file
save(beta, qc, pheno, file = "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00uthealth_brain-07312025.RData")